In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

car = pd.read_csv('quikr_car.csv')

old_car = car.copy()

car = car[car['year'].astype(str).str.isnumeric()]
car['year'] = car['year'].astype(str).str.strip()
car['year'] = car['year'].astype(int)

car = car[car['Price'] != 'Ask For Price']
car['Price'] = car['Price'].astype(str).str.replace(',', '')
car['Price'] = car['Price'].astype(int)


car['kms_driven'] = car['kms_driven'].astype(str).str.replace('kms','')
car['kms_driven'] = car['kms_driven'].astype(str).str.replace(',','')
car['kms_driven'] = car['kms_driven'].astype(str).str.strip()

car = car[car['kms_driven'].str.isnumeric()]
car['kms_driven'] = car['kms_driven'].astype(int)

print(car.shape)

(817, 6)


In [2]:
car.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,80000,45000,Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,425000,40,Diesel
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,325000,28000,Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,575000,36000,Diesel
6,Ford Figo,Ford,2012,175000,41000,Diesel


In [3]:
car['fuel_type'].unique()

array(['Petrol', 'Diesel', nan, 'LPG'], dtype=object)

In [4]:
car = car[~car['fuel_type'].isna()]

In [5]:
car['fuel_type'].unique()

array(['Petrol', 'Diesel', 'LPG'], dtype=object)

In [6]:
car['name'] = car['name'].str.split(' ').str.slice(0,3).str.join(' ')

In [7]:
car.reset_index(drop=True)

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing,Hyundai,2007,80000,45000,Petrol
1,Mahindra Jeep CL550,Mahindra,2006,425000,40,Diesel
2,Hyundai Grand i10,Hyundai,2014,325000,28000,Petrol
3,Ford EcoSport Titanium,Ford,2014,575000,36000,Diesel
4,Ford Figo,Ford,2012,175000,41000,Diesel
...,...,...,...,...,...,...
811,Maruti Suzuki Ritz,Maruti,2011,270000,50000,Petrol
812,Tata Indica V2,Tata,2009,110000,30000,Diesel
813,Toyota Corolla Altis,Toyota,2009,300000,132000,Petrol
814,Tata Zest XM,Tata,2018,260000,27000,Diesel


In [8]:
car = car[car['Price']<6000000]

In [9]:
car.reset_index(drop=True)

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing,Hyundai,2007,80000,45000,Petrol
1,Mahindra Jeep CL550,Mahindra,2006,425000,40,Diesel
2,Hyundai Grand i10,Hyundai,2014,325000,28000,Petrol
3,Ford EcoSport Titanium,Ford,2014,575000,36000,Diesel
4,Ford Figo,Ford,2012,175000,41000,Diesel
...,...,...,...,...,...,...
810,Maruti Suzuki Ritz,Maruti,2011,270000,50000,Petrol
811,Tata Indica V2,Tata,2009,110000,30000,Diesel
812,Toyota Corolla Altis,Toyota,2009,300000,132000,Petrol
813,Tata Zest XM,Tata,2018,260000,27000,Diesel


In [10]:
car.to_csv('Clean_car_data.csv',index=False)

In [11]:
X = car.drop('Price',axis=1)

In [12]:
y = car['Price']

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [14]:
cat_cols = ['name','company','fuel_type']
num_cols = ['year','kms_driven']

In [15]:
num_pipe = Pipeline([
        ('scaler',StandardScaler())
])
cat_pipe = Pipeline([
        ('ohe',OneHotEncoder(handle_unknown='ignore', drop='first'))
])

In [16]:
preprocessor = ColumnTransformer([
    ('num',num_pipe,num_cols),
    ('cat',cat_pipe,cat_cols)
])

In [17]:
pipeline = Pipeline([
    ('processor',preprocessor),
    ('regression', RandomForestRegressor())
])

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

In [20]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('processor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['year', 'kms_driven']),
                                                 ('cat',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['name', 'company',
                                                   'fuel_type'])])),
                ('regression', RandomForestRegressor())])

In [21]:
pred = pipeline.predict(X_test)

C:\Users\Arkaprava\anaconda3\envs\py310_env\lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [22]:
r2_score(y_test,pred)

0.7182529852821287

In [23]:
sample = pd.DataFrame([{
    'name': 'Renault Duster 85',
    'company': 'Renault',
    'year': 2015,
    'kms_driven': 715000,
    'fuel_type': 'Diesel'
}])

pred_price = pipeline.predict(sample)
print(pred_price)

[362672.02]


pandas.core.series.Series